# Beyond Retrieval Quality: How Embedding Architecture Causally Shapes Faithfulness in RAG Systems
**Author:** Shakhriyorbek Boltabaev | University of Szeged  
**Draft:** March 2026  

---

## Execution Strategy — Split Pipeline

This notebook is designed to run on **mixed infrastructure**:

| Phase | Where | Time Est. | Cost |
|-------|-------|-----------|------|
| **Phase A:** Embedding + FAISS indexing | 🏠 Local CPU (Ryzen 5) | ~6-8h overnight | Free |
| **Phase B:** Retrieval (top-k search) | 🏠 Local CPU | ~1-2h | Free |
| **Phase C:** GPT-4o-mini generation | 🏠 Local CPU (API calls) | ~2-3h | **~$5-6** |
| **Phase D:** DeBERTa-NLI faithfulness | 🏠 Local CPU | ~8-10h overnight | Free |
| **Phase E:** AlignScore evaluation | ☁️ Google Colab (T4 GPU) | ~1-2h | Free |
| **Phase F:** Llama-3 validation subset | ☁️ Google Colab (T4 GPU) | ~3-4h | Free |
| **Phase G:** Re-ranking + Analysis | 🏠 Local CPU | ~2-3h | Free |
| **Phase H:** Visualization + Export | 🏠 Local CPU | ~5min | Free |

**Total estimated cost: $5-6 API + free Colab**

Each phase saves checkpoints to disk so you can stop/resume freely.


---
## ☁️ Lambda Cloud Setup (run FIRST on a Lambda GPU instance)

**Check live prices yourself:** https://lambda.ai/service/gpu-cloud

### Which instance to pick
Your heaviest job is Llama-3-8B-Instruct (~16 GB VRAM in fp16). You do **not** need an A100 80GB or H100.

| Instance | VRAM | Good for | Notes |
|----------|------|----------|-------|
| **A10 (24 GB)** | 24 GB | Everything in this paper | Cheapest that fits Llama-3-8B comfortably |
| **A100 40GB** | 40 GB | Everything + headroom | Faster, more $/hr |
| RTX 6000 / A6000 | 48 GB | Overkill here | Only if A10/A100 unavailable |

**Recommendation:** start with the smallest instance that fits (A10 24GB). The whole GPU portion of this paper (AlignScore + Llama-3 subset + optionally embeddings/NLI) is ~10–12 GPU-hours. Use the **free $15 credit** first — it may cover the entire run.

### Workflow
1. Launch instance on Lambda → it comes pre-installed with Lambda Stack (Ubuntu + PyTorch + CUDA + cuDNN).
2. Open the Jupyter link Lambda gives you, upload this notebook.
3. Run the setup cell below, then Phases E and F (and optionally A–D to go faster than your Ryzen).
4. **Persist checkpoints**: Lambda instances are ephemeral. Either mount a Lambda persistent filesystem, or `rsync`/upload checkpoints to your own machine before terminating.
5. **Terminate the instance** when done — billing is per-minute and stops only on terminate, not on idle.


In [ ]:
# ════════════════════════════════════════════════════════════
# LAMBDA CLOUD SETUP
# Run this once at the start of a Lambda GPU session.
# ════════════════════════════════════════════════════════════
import subprocess, os, sys

# 1. Confirm GPU is visible
print('=== GPU CHECK ===')
subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.used',
                '--format=csv,noheader'], check=False)

import torch
assert torch.cuda.is_available(), 'No GPU detected! Check your Lambda instance.'
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'\n✓ GPU: {gpu_name} ({vram_gb:.0f} GB VRAM)')
if vram_gb < 15:
    print('⚠️  Warning: <15 GB VRAM. Llama-3-8B may OOM. Consider a larger instance,')
    print('   or run Llama-3 in 8-bit (set LOAD_LLAMA_8BIT=True in Phase F).')
else:
    print('✓ Sufficient VRAM for Llama-3-8B-Instruct in fp16.')

# 2. Lambda Stack already has torch/CUDA. Install only what's missing.
print('\n=== INSTALLING PROJECT DEPS ===')
pkgs = ['sentence-transformers', 'faiss-gpu', 'transformers', 'datasets',
        'openai', 'accelerate', 'bitsandbytes', 'ranx', 'alignscore',
        'seaborn', 'tqdm']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=False)
print('✓ Dependencies installed.')

# 3. Persistent checkpoint directory
#    Lambda persistent filesystems mount under /home/ubuntu or a path you set.
#    If you attached a Lambda filesystem named e.g. 'rag-data', point here:
LAMBDA_PERSISTENT = os.getenv('LAMBDA_FS_PATH', '/home/ubuntu/rag_faith_checkpoints')
os.makedirs(LAMBDA_PERSISTENT, exist_ok=True)
print(f'\n✓ Checkpoint dir: {LAMBDA_PERSISTENT}')
print('  (Upload your Phase A–D checkpoints here before running E/F,')
print('   and download results from here before terminating the instance.)')

# 4. API keys (set these as environment variables or paste here)
if not os.getenv('OPENAI_API_KEY'):
    print('\n⚠️  OPENAI_API_KEY not set (only needed if running Phase C here).')
if not os.getenv('HF_TOKEN'):
    print('⚠️  HF_TOKEN not set — required to download Llama-3 (gated model).')
    print('   Get one at https://huggingface.co/settings/tokens and accept the')
    print('   Llama-3 license at https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct')

print('\n✓ Lambda setup complete. Proceed to Cell 2 (config), then Phases E/F.')
print('  REMINDER: terminate the instance when done to stop billing.')

---
## Cell 1 — Environment Setup & Dependency Installation

Run once. Works on both local and Colab.

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

# Core packages (both local + Colab)
!pip install -q sentence-transformers faiss-cpu transformers datasets
!pip install -q openai torch accelerate
!pip install -q ranx matplotlib seaborn scipy pandas numpy tqdm

# AlignScore — install on Colab only (heavy, needs GPU)
if IN_COLAB:
    !pip install -q alignscore
    print('✓ AlignScore installed (Colab GPU mode)')
else:
    print('⏭ Skipping AlignScore install (will run on Colab later)')

print(f'\n✓ Environment: {"Google Colab" if IN_COLAB else "Local CPU"}')

---
## Cell 2 — Imports, Configuration & API Cost Tracker

In [ ]:
import os
import sys
import json
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm
from scipy import stats
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional

import torch
import faiss
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    AutoModelForCausalLM
)
from datasets import load_dataset
import openai

# ── Runtime Detection ─────────────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── Checkpoint Directory ──────────────────────────────────────────
# On Colab, mount Google Drive for persistent checkpoints
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_DIR = Path('/content/drive/MyDrive/rag_faith_checkpoints')
elif os.getenv('LAMBDA_FS_PATH') or os.path.isdir('/home/ubuntu/rag_faith_checkpoints'):
    # Lambda Cloud persistent filesystem
    CHECKPOINT_DIR = Path(os.getenv('LAMBDA_FS_PATH', '/home/ubuntu/rag_faith_checkpoints'))
else:
    CHECKPOINT_DIR = Path('./rag_faith_checkpoints')
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# ── Configuration ─────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

TOP_K          = 5
CHUNK_SIZE     = 256
CHUNK_OVERLAP  = 32
N_QUERIES      = 1000    # Full run: 1000. Quick test: 50
LAMBDA_RERANK  = 0.6
LLAMA_SUBSET   = 300     # Llama-3 validation: 300 per dataset on Colab

# ── OpenAI API ────────────────────────────────────────────────────
openai.api_key = os.getenv('OPENAI_API_KEY', 'YOUR_KEY_HERE')

# ── API Cost Tracker ──────────────────────────────────────────────
class CostTracker:
    """Track OpenAI API costs in real time."""
    # GPT-4o-mini pricing (as of March 2026)
    INPUT_COST_PER_1M  = 0.15   # $/1M input tokens
    OUTPUT_COST_PER_1M = 0.60   # $/1M output tokens

    def __init__(self):
        self.total_input_tokens = 0
        self.total_output_tokens = 0
        self.total_requests = 0
        self.errors = 0

    def log(self, usage):
        """Log usage from an OpenAI API response."""
        self.total_input_tokens += usage.prompt_tokens
        self.total_output_tokens += usage.completion_tokens
        self.total_requests += 1

    def log_error(self):
        self.errors += 1

    @property
    def cost(self) -> float:
        input_cost  = (self.total_input_tokens / 1_000_000) * self.INPUT_COST_PER_1M
        output_cost = (self.total_output_tokens / 1_000_000) * self.OUTPUT_COST_PER_1M
        return input_cost + output_cost

    def summary(self) -> str:
        return (
            f'API Cost Summary:\n'
            f'  Requests:      {self.total_requests:,}\n'
            f'  Input tokens:  {self.total_input_tokens:,}\n'
            f'  Output tokens: {self.total_output_tokens:,}\n'
            f'  Errors:        {self.errors}\n'
            f'  ──────────────────────\n'
            f'  Total cost:    ${self.cost:.4f}\n'
        )

cost_tracker = CostTracker()

# ── Checkpoint Helpers ────────────────────────────────────────────
def save_checkpoint(name: str, data):
    """Save intermediate results to disk."""
    path = CHECKPOINT_DIR / f'{name}.pkl'
    with open(path, 'wb') as f:
        pickle.dump(data, f)
    print(f'  💾 Saved checkpoint: {path}')

def load_checkpoint(name: str):
    """Load checkpoint if it exists, otherwise return None."""
    path = CHECKPOINT_DIR / f'{name}.pkl'
    if path.exists():
        with open(path, 'rb') as f:
            data = pickle.load(f)
        print(f'  ✓ Loaded checkpoint: {path}')
        return data
    return None

def checkpoint_exists(name: str) -> bool:
    return (CHECKPOINT_DIR / f'{name}.pkl').exists()

print(f'Device: {DEVICE}')
print(f'Checkpoints: {CHECKPOINT_DIR}')
print(f'Colab: {IN_COLAB}')
print(f'N_QUERIES: {N_QUERIES}')
print('✓ Configuration loaded.')

---
## Cell 3 — Embedding Model Definitions (§4.1, Table 1)
Six models covering contrastive, instruction-tuned, and multilingual training paradigms.

In [ ]:
@dataclass
class EmbeddingModelConfig:
    name: str
    hf_id: str
    paradigm: str       # contrastive | instruction-tuned | multilingual
    dimension: int
    instruction: Optional[str] = None
    color: str = '#4C72B0'

EMBEDDING_MODELS = [
    EmbeddingModelConfig(
        name='all-mpnet-base-v2',
        hf_id='sentence-transformers/all-mpnet-base-v2',
        paradigm='contrastive', dimension=768, color='#E74C3C'
    ),
    EmbeddingModelConfig(
        name='GTE-large',
        hf_id='thenlper/gte-large',
        paradigm='contrastive', dimension=1024, color='#E67E22'
    ),
    EmbeddingModelConfig(
        name='BGE-M3',
        hf_id='BAAI/bge-m3',
        paradigm='multilingual', dimension=1024, color='#8E44AD'
    ),
    EmbeddingModelConfig(
        name='E5-large-instruct',
        hf_id='intfloat/e5-large-instruct',
        paradigm='instruction-tuned', dimension=1024,
        instruction='Represent this sentence for searching relevant passages: ',
        color='#27AE60'
    ),
    EmbeddingModelConfig(
        name='Instructor-XL',
        hf_id='hkunlp/instructor-xl',
        paradigm='instruction-tuned', dimension=768,
        instruction='Represent the question for retrieving relevant documents: ',
        color='#2ECC71'
    ),
    EmbeddingModelConfig(
        name='text-embedding-3-small',
        hf_id='openai',
        paradigm='contrastive', dimension=1536, color='#3498DB'
    ),
]

print('Embedding model configs:')
for m in EMBEDDING_MODELS:
    print(f'  {m.name:30s} | {m.paradigm:18s} | dim={m.dimension}')

---
## Cell 4 — Embedding Model Loader (with checkpointing)

In [ ]:
class EmbeddingModelWrapper:
    """
    Unified wrapper for all embedding models.
    Handles HuggingFace SentenceTransformer and OpenAI API models.
    """
    def __init__(self, config: EmbeddingModelConfig):
        self.config = config
        self.model = None
        if config.hf_id != 'openai':
            print(f'Loading {config.name}...')
            self.model = SentenceTransformer(config.hf_id, device=DEVICE)
            print(f'  ✓ Loaded ({config.dimension}d)')

    def encode(self, texts: List[str], is_query: bool = False,
               batch_size: int = 32) -> np.ndarray:
        """Encode texts to normalized embedding vectors."""
        if self.config.hf_id == 'openai':
            return self._encode_openai(texts)
        if self.config.instruction and is_query:
            texts = [self.config.instruction + t for t in texts]
        embeddings = self.model.encode(
            texts, normalize_embeddings=True,
            batch_size=batch_size, show_progress_bar=False
        )
        return embeddings

    def _encode_openai(self, texts: List[str]) -> np.ndarray:
        """Encode via OpenAI API (batched, with cost tracking)."""
        embeddings = []
        # Batch in groups of 100 for efficiency
        for i in range(0, len(texts), 100):
            batch = [t[:8000] for t in texts[i:i+100]]
            resp = openai.embeddings.create(
                model='text-embedding-3-small', input=batch
            )
            for item in resp.data:
                emb = np.array(item.embedding, dtype=np.float32)
                emb = emb / np.linalg.norm(emb)
                embeddings.append(emb)
        return np.array(embeddings)


def load_models_for_phase(model_names: List[str] = None) -> Dict[str, EmbeddingModelWrapper]:
    """
    Load specific models. On CPU, load one at a time to save RAM.
    Pass model_names=None to load all.
    """
    models = {}
    for cfg in EMBEDDING_MODELS:
        if model_names and cfg.name not in model_names:
            continue
        try:
            models[cfg.name] = EmbeddingModelWrapper(cfg)
        except Exception as e:
            print(f'  ✗ Could not load {cfg.name}: {e}')
    print(f'\nLoaded {len(models)} embedding models.')
    return models


# On local CPU: load models ONE AT A TIME in Phase A to avoid OOM
# On Colab GPU: can load all at once
print('Model loader ready.')
print(f'  Tip: On local CPU, run Phase A model-by-model.')
print(f'  Tip: On Colab GPU, load all with load_models_for_phase()')

---
## Cell 5 — Dataset Loading (§4.2)
NQ [17], HotpotQA [18], QASPER [19] — fixed loaders for HuggingFace API.

In [ ]:
@dataclass
class QASample:
    query_id: str
    question: str
    answer: str
    gold_context: str
    corpus_chunks: List[str] = field(default_factory=list)
    dataset: str = ''


def load_nq(n: int = N_QUERIES) -> List[QASample]:
    """Load Natural Questions [17] — single-hop factoid QA."""
    cached = load_checkpoint(f'dataset_nq_{n}')
    if cached: return cached

    print('Loading Natural Questions...')
    ds = load_dataset('natural_questions', split='validation', streaming=True)
    samples = []
    for i, item in enumerate(ds):
        if len(samples) >= n: break
        answer_text = None
        for sa in item['annotations']['short_answers']:
            if sa['text']:
                answer_text = sa['text'][0]
                break
        if not answer_text:
            continue
        tokens = item['document']['tokens']
        text_tokens = [t for t, h in zip(tokens['token'], tokens['is_html']) if not h]
        context = ' '.join(text_tokens[:500])
        samples.append(QASample(
            query_id=f'nq_{len(samples)}',
            question=item['question']['text'],
            answer=answer_text,
            gold_context=context,
            dataset='NQ'
        ))
    print(f'  ✓ Loaded {len(samples)} NQ samples')
    save_checkpoint(f'dataset_nq_{n}', samples)
    return samples


def load_hotpotqa(n: int = N_QUERIES) -> List[QASample]:
    """Load HotpotQA [18] — multi-hop reasoning QA."""
    cached = load_checkpoint(f'dataset_hotpot_{n}')
    if cached: return cached

    print('Loading HotpotQA...')
    ds = load_dataset('hotpot_qa', 'distractor', split='validation', streaming=True)
    samples = []
    for i, item in enumerate(ds):
        if len(samples) >= n: break
        sf = item['supporting_facts']
        ctx = item['context']
        gold_sents = []
        for title, sid in zip(sf['title'], sf['sent_id']):
            if title in ctx['title']:
                idx = ctx['title'].index(title)
                if sid < len(ctx['sentences'][idx]):
                    gold_sents.append(ctx['sentences'][idx][sid])
        if not gold_sents: continue
        corpus = []
        for sents in ctx['sentences']:
            corpus.extend(sents)
        samples.append(QASample(
            query_id=f'hotpot_{len(samples)}',
            question=item['question'],
            answer=item['answer'],
            gold_context=' '.join(gold_sents),
            corpus_chunks=corpus,
            dataset='HotpotQA'
        ))
    print(f'  ✓ Loaded {len(samples)} HotpotQA samples')
    save_checkpoint(f'dataset_hotpot_{n}', samples)
    return samples


def load_qasper(n: int = N_QUERIES) -> List[QASample]:
    """Load QASPER [19] — scientific paper QA."""
    cached = load_checkpoint(f'dataset_qasper_{n}')
    if cached: return cached

    print('Loading QASPER...')
    try:
        ds = load_dataset('allenai/qasper', split='validation', trust_remote_code=True)
    except Exception:
        ds = load_dataset('allenai/qasper', split='validation')
    samples = []
    for paper in ds:
        if len(samples) >= n: break
        qas = paper['qas']
        for q_idx in range(len(qas['question'])):
            if len(samples) >= n: break
            question = qas['question'][q_idx]
            answers_data = qas['answers'][q_idx]
            answer, evidence_text = None, ''
            for ans in answers_data['answer']:
                if ans.get('free_form_answer'):
                    answer = ans['free_form_answer']
                    evidence_text = ' '.join(ans.get('evidence', []))
                    break
                elif ans.get('extractive_spans'):
                    answer = ' '.join(ans['extractive_spans'])
                    evidence_text = ' '.join(ans.get('evidence', []))
                    break
            if not answer: continue
            corpus = []
            for section_paragraphs in paper['full_text']['paragraphs']:
                corpus.extend(section_paragraphs)
            gold_ctx = evidence_text if evidence_text else ' '.join(corpus[:3])
            samples.append(QASample(
                query_id=f'qasper_{len(samples)}',
                question=question, answer=answer,
                gold_context=gold_ctx, corpus_chunks=corpus,
                dataset='QASPER'
            ))
    print(f'  ✓ Loaded {len(samples)} QASPER samples')
    save_checkpoint(f'dataset_qasper_{n}', samples)
    return samples


# Load all datasets (cached after first run)
datasets_dict = {
    'NQ': load_nq(N_QUERIES),
    'HotpotQA': load_hotpotqa(N_QUERIES),
    'QASPER': load_qasper(N_QUERIES),
}
print(f'\nDatasets: {{", ".join(f"{k}: {len(v)}" for k, v in datasets_dict.items())}}')

---
## Cell 6 — Text Chunking & FAISS Indexing (§5.1)
Chunk size: 256 tokens, overlap: 32 tokens. FAISS for nearest-neighbor search [14].

In [ ]:
from transformers import AutoTokenizer

TOKENIZER = AutoTokenizer.from_pretrained("bert-base-uncased")


def chunk_text(text: str, chunk_size: int = CHUNK_SIZE,
               overlap: int = CHUNK_OVERLAP) -> List[str]:
    """Split text into overlapping token-based chunks (§5.1)."""
    tokens = TOKENIZER.tokenize(text)
    chunks, start = [], 0
    while start < len(tokens):
        end = min(start + chunk_size, len(tokens))
        chunk_tokens = tokens[start:end]
        chunk_text = TOKENIZER.convert_tokens_to_string(chunk_tokens)
        chunks.append(chunk_text)
        if end == len(tokens):
            break
        start += chunk_size - overlap
    return chunks


class FAISSIndex:
    """
    FAISS-based dense retrieval index [14].
    Supports exact inner product search (cosine similarity on normalized vectors).
    """
    def __init__(self, dimension: int):
        self.index = faiss.IndexFlatIP(dimension)  # Inner product = cosine on normalized vecs
        self.chunks: List[str] = []
        self.dimension = dimension

    def add(self, chunks: List[str], embeddings: np.ndarray):
        assert embeddings.shape[1] == self.dimension, \
            f"Dimension mismatch: expected {self.dimension}, got {embeddings.shape[1]}"
        self.chunks.extend(chunks)
        self.index.add(embeddings.astype(np.float32))

    def search(self, query_emb: np.ndarray, k: int = TOP_K) -> List[Tuple[str, float]]:
        """Return top-k (chunk, score) pairs."""
        query_emb = query_emb.astype(np.float32).reshape(1, -1)
        scores, indices = self.index.search(query_emb, k)
        return [(self.chunks[i], float(scores[0][j]))
                for j, i in enumerate(indices[0]) if i >= 0]

    def __len__(self):
        return self.index.ntotal


def build_index(samples: List[QASample], model_wrapper: EmbeddingModelWrapper) -> FAISSIndex:
    """Chunk all corpus documents and build FAISS index."""
    dim = model_wrapper.config.dimension
    idx = FAISSIndex(dimension=dim)
    all_chunks, all_texts = [], []

    for sample in samples:
        corpus = sample.corpus_chunks if sample.corpus_chunks else [sample.gold_context]
        for doc in corpus:
            if not doc.strip(): continue
            chunks = chunk_text(doc)
            all_chunks.extend(chunks)
            # Track which query_id each chunk belongs to (for relevance labels)
            all_texts.extend([(sample.query_id, c) for c in chunks])

    # Batch-encode all chunks
    print(f"  Encoding {len(all_chunks)} chunks with {model_wrapper.config.name}...")
    batch_size = 128
    all_embs = []
    for i in tqdm(range(0, len(all_chunks), batch_size)):
        batch = all_chunks[i:i+batch_size]
        embs = model_wrapper.encode(batch, is_query=False)
        all_embs.append(embs)
    all_embs = np.vstack(all_embs)

    idx.add(all_chunks, all_embs)
    print(f"  ✓ Index built: {len(idx)} vectors")
    return idx


print("Chunking and indexing utilities ready.")

---
## 🏠 PHASE A — Embedding & Indexing (Local CPU, overnight)

Run this on your Ryzen. Loads one model at a time to save RAM.  
Saves embeddings + FAISS indexes as checkpoints.  
**Estimated time: 6-8 hours total (run overnight).**

In [ ]:
def run_phase_a(datasets_dict, model_names=None):
    """
    PHASE A: Build FAISS indexes for all model × dataset combos.
    Loads models ONE AT A TIME to stay within 16GB RAM.
    Saves: checkpoints/index_{model}_{dataset}.pkl
           checkpoints/retrieval_{model}_{dataset}.pkl
    """
    configs = EMBEDDING_MODELS if not model_names else [
        c for c in EMBEDDING_MODELS if c.name in model_names
    ]

    for cfg in configs:
        print(f'\n{"="*60}')
        print(f'Model: {cfg.name} ({cfg.paradigm})')

        # Check if already done
        all_done = all(
            checkpoint_exists(f'retrieval_{cfg.name}_{ds}')
            for ds in datasets_dict
        )
        if all_done:
            print(f'  ⏭ All datasets already indexed. Skipping.')
            continue

        # Load model
        t0 = time.time()
        wrapper = EmbeddingModelWrapper(cfg)

        for ds_name, samples in datasets_dict.items():
            ck_name = f'retrieval_{cfg.name}_{ds_name}'
            if checkpoint_exists(ck_name):
                print(f'  ⏭ {ds_name} already done.')
                continue

            print(f'\n  Indexing {ds_name} ({len(samples)} samples)...')
            index = build_index(samples, wrapper)

            # Retrieve top-k for all queries
            print(f'  Retrieving top-{TOP_K} for all queries...')
            retrievals = []
            for sample in tqdm(samples, desc=f'    Retrieval [{cfg.name}]'):
                q_emb = wrapper.encode([sample.question], is_query=True)
                results = index.search(q_emb[0], k=TOP_K)
                retrievals.append({
                    'query_id': sample.query_id,
                    'question': sample.question,
                    'answer': sample.answer,
                    'gold_context': sample.gold_context,
                    'retrieved_chunks': [c for c, _ in results],
                    'retrieval_scores': [s for _, s in results],
                })
            save_checkpoint(ck_name, retrievals)
            print(f'  ✓ {ds_name}: {len(retrievals)} queries retrieved')

        # Free memory before loading next model
        del wrapper
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        import gc; gc.collect()
        print(f'  ⏱ Model done in {time.time()-t0:.0f}s')

    print(f'\n{"="*60}')
    print('✓ PHASE A complete!')


# ── RUN PHASE A ──────────────────────────────────────────────────
# On first run, do all 6 models. Or start with 3 core models:
# run_phase_a(datasets_dict, ['all-mpnet-base-v2', 'E5-large-instruct', 'BGE-M3'])
run_phase_a(datasets_dict)

---
## 🏠 PHASE B — Retrieval Quality Metrics (Local CPU, ~30 min)
NDCG@5, Recall@5, MRR using `ranx`. Quick CPU job.

In [ ]:
from ranx import Qrels, Run, evaluate

def run_phase_b(datasets_dict):
    """Compute retrieval quality from saved retrievals."""
    if checkpoint_exists('retrieval_quality_all'):
        return load_checkpoint('retrieval_quality_all')

    results = {}  # {model: {dataset: {metric: val}}}
    for cfg in EMBEDDING_MODELS:
        results[cfg.name] = {}
        for ds_name, samples in datasets_dict.items():
            retrievals = load_checkpoint(f'retrieval_{cfg.name}_{ds_name}')
            if not retrievals:
                print(f'  ✗ Missing retrieval for {cfg.name}/{ds_name}. Run Phase A first.')
                continue

            # Build qrels: gold chunk = first chunk of gold_context
            qrels_dict, run_dict = {}, {}
            for r, s in zip(retrievals, samples):
                gold_chunks = chunk_text(s.gold_context)
                gold_id = gold_chunks[0] if gold_chunks else s.gold_context
                qrels_dict[r['query_id']] = {gold_id: 1}
                run_dict[r['query_id']] = {
                    c: sc for c, sc in zip(r['retrieved_chunks'], r['retrieval_scores'])
                }

            qrels = Qrels(qrels_dict)
            run = Run(run_dict)
            metrics = evaluate(qrels, run, ['ndcg@5', 'recall@5', 'mrr@5'])
            results[cfg.name][ds_name] = {
                'NDCG@5': round(metrics['ndcg@5'], 4),
                'Recall@5': round(metrics['recall@5'], 4),
                'MRR@5': round(metrics['mrr@5'], 4),
            }
            print(f'  [{ds_name}] {cfg.name}: {results[cfg.name][ds_name]}')

    save_checkpoint('retrieval_quality_all', results)
    return results

retrieval_quality = run_phase_b(datasets_dict)
print('\n✓ PHASE B complete!')

---
## 🏠 PHASE C — GPT-4o-mini Answer Generation (Local CPU, ~$5-6)

Runs on any machine — just needs internet + API key.  
**Cost tracker shows running total. Budget ~$5-6 for 18,000 queries.**  
Saves every 100 queries as checkpoint for fault tolerance.

In [ ]:
RAG_PROMPT_TEMPLATE = """You are a helpful assistant. Answer the question using ONLY the provided context.
Do not use any external knowledge. If the context does not contain enough information, say "I cannot answer based on the provided context."

Context:
{context}

Question: {question}

Answer:"""


def generate_gpt4o_mini(question: str, chunks: List[str]) -> Tuple[str, dict]:
    """Generate answer with GPT-4o-mini. Returns (answer, usage_dict)."""
    context = '\n\n'.join([f'[Chunk {i+1}]: {c}' for i, c in enumerate(chunks)])
    prompt = RAG_PROMPT_TEMPLATE.format(context=context, question=question)
    try:
        response = openai.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0,
            max_tokens=256
        )
        cost_tracker.log(response.usage)
        return response.choices[0].message.content.strip(), {
            'input_tokens': response.usage.prompt_tokens,
            'output_tokens': response.usage.completion_tokens,
        }
    except Exception as e:
        cost_tracker.log_error()
        return f'[ERROR: {e}]', {}


def run_phase_c(datasets_dict):
    """
    PHASE C: Generate answers for all model × dataset combos using GPT-4o-mini.
    Saves incrementally every 100 queries.
    """
    for cfg in EMBEDDING_MODELS:
        for ds_name in datasets_dict:
            ck_name = f'generated_gpt4o_{cfg.name}_{ds_name}'
            if checkpoint_exists(ck_name):
                existing = load_checkpoint(ck_name)
                print(f'  ⏭ {cfg.name}/{ds_name}: {len(existing)} already generated')
                continue

            retrievals = load_checkpoint(f'retrieval_{cfg.name}_{ds_name}')
            if not retrievals:
                print(f'  ✗ No retrieval for {cfg.name}/{ds_name}. Run Phase A.')
                continue

            print(f'\n  Generating [{cfg.name}] [{ds_name}]...')
            generations = []
            for i, r in enumerate(tqdm(retrievals, desc=f'    GPT-4o')):
                answer, usage = generate_gpt4o_mini(
                    r['question'], r['retrieved_chunks']
                )
                generations.append({
                    **r,
                    'generated_answer': answer,
                    'generator': 'gpt4o-mini',
                    'api_usage': usage,
                })
                # Incremental save every 100
                if (i + 1) % 100 == 0:
                    save_checkpoint(ck_name + '_partial', generations)
                    print(f'      💰 Running cost: ${cost_tracker.cost:.4f}')

            save_checkpoint(ck_name, generations)
            print(f'  ✓ {cfg.name}/{ds_name}: {len(generations)} answers')

    print(f'\n{cost_tracker.summary()}')
    print('✓ PHASE C complete!')


# ── RUN ──
run_phase_c(datasets_dict)

---
## 🏠 PHASE D — DeBERTa-NLI Faithfulness (Local CPU, overnight)

Runs DeBERTa-v3-large NLI [20] on CPU. ~5-10 pairs/second.  
**Estimated: 8-10 hours for 18,000 queries. Run overnight.**

In [ ]:
class NLIScorer:
    """DeBERTa-v3-large NLI scorer [20]. Works on CPU."""
    def __init__(self):
        print('Loading DeBERTa-v3-large NLI [20]...')
        model_name = 'cross-encoder/nli-deberta-v3-large'
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name
        ).to(DEVICE).eval()
        print(f'  ✓ Loaded on {DEVICE}')

    def entailment_score(self, premise: str, hypothesis: str) -> float:
        """P(entailment | premise, hypothesis)."""
        inputs = self.tokenizer(
            premise, hypothesis, return_tensors='pt',
            truncation=True, max_length=512
        ).to(DEVICE)
        with torch.no_grad():
            logits = self.model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)[0]
        return probs[2].item()  # entailment


def run_phase_d(datasets_dict):
    """
    PHASE D: Score NLI faithfulness for all generated answers.
    """
    nli = NLIScorer()

    for cfg in EMBEDDING_MODELS:
        for ds_name in datasets_dict:
            ck_out = f'nli_scores_{cfg.name}_{ds_name}'
            if checkpoint_exists(ck_out):
                print(f'  ⏭ {cfg.name}/{ds_name} NLI already scored.')
                continue

            ck_gen = f'generated_gpt4o_{cfg.name}_{ds_name}'
            generations = load_checkpoint(ck_gen)
            if not generations:
                print(f'  ✗ No generations for {cfg.name}/{ds_name}. Run Phase C.')
                continue

            print(f'\n  NLI scoring [{cfg.name}] [{ds_name}]...')
            scores = []
            for g in tqdm(generations, desc=f'    NLI'):
                context = ' '.join(g['retrieved_chunks'])
                score = nli.entailment_score(
                    premise=context,
                    hypothesis=g['generated_answer']
                )
                scores.append({
                    'query_id': g['query_id'],
                    'nli_entailment': score,
                })
            save_checkpoint(ck_out, scores)
            print(f'  ✓ {cfg.name}/{ds_name}: mean NLI = {np.mean([s["nli_entailment"] for s in scores]):.4f}')

    print('\n✓ PHASE D complete!')


# ── RUN ──
run_phase_d(datasets_dict)

---
## ☁️ PHASE E — AlignScore Evaluation (Colab T4 GPU, ~1-2h)

**Run this cell on Google Colab only.**  
Upload checkpoints from Phase C to Google Drive first.  
AlignScore [16] needs GPU for reasonable speed.

In [ ]:
def run_phase_e(datasets_dict):
    """PHASE E: AlignScore faithfulness (Colab GPU)."""
    if not IN_COLAB and DEVICE == 'cpu':
        print('⚠️  AlignScore is very slow on CPU. Run on Colab instead.')
        print('   To run anyway (hours), set FORCE_CPU_ALIGNSCORE = True')
        if not globals().get('FORCE_CPU_ALIGNSCORE', False):
            return

    from alignscore import AlignScore
    scorer = AlignScore(
        model='roberta-large', batch_size=32,
        device=DEVICE, evaluation_mode='nli_sp'
    )

    for cfg in EMBEDDING_MODELS:
        for ds_name in datasets_dict:
            ck_out = f'align_scores_{cfg.name}_{ds_name}'
            if checkpoint_exists(ck_out):
                print(f'  ⏭ {cfg.name}/{ds_name} AlignScore done.')
                continue

            generations = load_checkpoint(f'generated_gpt4o_{cfg.name}_{ds_name}')
            if not generations:
                print(f'  ✗ Missing generations for {cfg.name}/{ds_name}')
                continue

            print(f'\n  AlignScore [{cfg.name}] [{ds_name}]...')
            contexts = [' '.join(g['retrieved_chunks']) for g in generations]
            claims = [g['generated_answer'] for g in generations]
            raw_scores = scorer.score(contexts=contexts, claims=claims)

            scores = [
                {'query_id': g['query_id'], 'align_score': float(s)}
                for g, s in zip(generations, raw_scores)
            ]
            save_checkpoint(ck_out, scores)
            print(f'  ✓ mean AlignScore = {np.mean(raw_scores):.4f}')

    print('\n✓ PHASE E complete!')


run_phase_e(datasets_dict)

---
## ☁️ PHASE F — Llama-3 Validation Subset (Colab T4, ~3-4h)

**Run on Colab only.** Tests H3 (generator independence).  
Runs 300 queries per dataset (900 total) instead of full 3,000.

In [ ]:
class Llama3Generator:
    """Llama-3-8B-Instruct [§4.4]. Requires GPU."""
    def __init__(self):
        print('Loading Llama-3-8B-Instruct...')
        model_id = 'meta-llama/Meta-Llama-3-8B-Instruct'
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id, token=os.getenv('HF_TOKEN')
        )
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, torch_dtype=torch.float16,
            device_map='auto', token=os.getenv('HF_TOKEN')
        )
        self.model.eval()
        print('  ✓ Loaded')

    def generate(self, question: str, chunks: List[str]) -> str:
        context = '\n\n'.join([f'[Chunk {i+1}]: {c}' for i, c in enumerate(chunks)])
        prompt = RAG_PROMPT_TEMPLATE.format(context=context, question=question)
        inputs = self.tokenizer(
            prompt, return_tensors='pt', truncation=True, max_length=2048
        ).to(self.model.device)
        with torch.no_grad():
            output = self.model.generate(
                **inputs, max_new_tokens=256,
                temperature=1.0, do_sample=False
            )
        decoded = self.tokenizer.decode(output[0], skip_special_tokens=True)
        return decoded[len(prompt):].strip()


def run_phase_f(datasets_dict):
    """PHASE F: Llama-3 validation on subset."""
    if DEVICE != 'cuda':
        print('⚠️  Llama-3 requires GPU. Run on Colab.')
        return

    llama = Llama3Generator()
    nli = NLIScorer()  # For immediate NLI scoring

    # Use 3 core models for validation
    val_models = ['all-mpnet-base-v2', 'E5-large-instruct', 'BGE-M3']

    for model_name in val_models:
        for ds_name in datasets_dict:
            ck_out = f'llama3_validation_{model_name}_{ds_name}'
            if checkpoint_exists(ck_out):
                print(f'  ⏭ {model_name}/{ds_name} Llama-3 done.')
                continue

            retrievals = load_checkpoint(f'retrieval_{model_name}_{ds_name}')
            if not retrievals:
                continue

            subset = retrievals[:LLAMA_SUBSET]
            print(f'\n  Llama-3 [{model_name}] [{ds_name}] ({len(subset)} queries)...')

            results = []
            for r in tqdm(subset, desc=f'    Llama-3'):
                answer = llama.generate(r['question'], r['retrieved_chunks'])
                context = ' '.join(r['retrieved_chunks'])
                nli_score = nli.entailment_score(context, answer)
                results.append({
                    'query_id': r['query_id'],
                    'generated_answer': answer,
                    'nli_entailment': nli_score,
                    'generator': 'llama3',
                })

            save_checkpoint(ck_out, results)
            print(f'  ✓ mean NLI = {np.mean([r["nli_entailment"] for r in results]):.4f}')

    print('\n✓ PHASE F complete!')


run_phase_f(datasets_dict)

---
## 🏠 PHASE G — RFG Computation, Re-ranking & Mechanistic Analysis

Assembles all checkpoints into the final results DataFrame.  
Computes RFG (Eq. 2), runs re-ranking ablation (Eq. 4), and ESA correlation (Eq. 3).

In [ ]:
def compute_rfg(retrieval_quality: float, faithfulness: float) -> float:
    """RFG(E) = NDCG@5(E) − Faithfulness(E)  [Eq. 2]"""
    return round(max(0, retrieval_quality) - max(0, faithfulness), 4)


def assemble_results() -> pd.DataFrame:
    """
    Combine all phase checkpoints into a single results DataFrame.
    """
    if checkpoint_exists('final_results_df'):
        return load_checkpoint('final_results_df')

    retrieval_quality = load_checkpoint('retrieval_quality_all')
    if not retrieval_quality:
        print('✗ Run Phase B first.'); return None

    rows = []
    for cfg in EMBEDDING_MODELS:
        for ds_name in ['NQ', 'HotpotQA', 'QASPER']:
            rq = retrieval_quality.get(cfg.name, {}).get(ds_name)
            if not rq: continue

            # NLI scores
            nli_scores = load_checkpoint(f'nli_scores_{cfg.name}_{ds_name}')
            # AlignScores
            align_scores = load_checkpoint(f'align_scores_{cfg.name}_{ds_name}')

            mean_nli = np.mean([s['nli_entailment'] for s in nli_scores]) if nli_scores else None
            mean_align = np.mean([s['align_score'] for s in align_scores]) if align_scores else None

            # Use whichever faithfulness scores we have
            if mean_align is not None and mean_nli is not None:
                mean_faith = (mean_align + mean_nli) / 2
            elif mean_nli is not None:
                mean_faith = mean_nli
            elif mean_align is not None:
                mean_faith = mean_align
            else:
                continue

            rows.append({
                'model': cfg.name,
                'paradigm': cfg.paradigm,
                'dataset': ds_name,
                'NDCG@5': rq['NDCG@5'],
                'Recall@5': rq['Recall@5'],
                'MRR@5': rq['MRR@5'],
                'align_score': mean_align,
                'nli_entailment': mean_nli,
                'mean_faithfulness': mean_faith,
                'RFG': compute_rfg(rq['NDCG@5'], mean_faith),
            })

    df = pd.DataFrame(rows)
    save_checkpoint('final_results_df', df)
    print(f'\nAssembled {len(df)} result rows.')
    print(df.groupby('model')[['NDCG@5', 'mean_faithfulness', 'RFG']].mean().round(3))
    return df


results_df = assemble_results()
if results_df is not None:
    print('\n✓ PHASE G: Results assembled!')

---
## 🏠 PHASE G.2 — Faithfulness-Aware Re-ranking Experiment (§4.6)

In [ ]:
def faithfulness_aware_rerank(
    query: str,
    retrieved_chunks: List[Tuple[str, float]],  # (chunk_text, cosine_score)
    nli_model,
    nli_tokenizer,
    lambda_mix: float = LAMBDA_RERANK
) -> List[Tuple[str, float]]:
    """
    Re-rank retrieved chunks by combining cosine similarity
    and NLI entailment probability (§4.6).

    score(dᵢ, q) = λ · cos(E(dᵢ), E(q)) + (1−λ) · NLI(dᵢ, q)

    Args:
        query:            The original question
        retrieved_chunks: List of (chunk_text, cosine_score) from FAISS
        nli_model:        DeBERTa-NLI model [20]
        nli_tokenizer:    Corresponding tokenizer
        lambda_mix:       Mixing parameter (default 0.6 favors cosine)

    Returns:
        Re-ranked list of (chunk_text, combined_score)
    """
    reranked = []
    for chunk_text, cos_score in retrieved_chunks:
        # NLI: P(chunk entails answer to query)
        # Here we use query as hypothesis, chunk as premise
        inputs = nli_tokenizer(
            chunk_text, query,
            return_tensors="pt", truncation=True, max_length=512
        ).to(DEVICE)
        with torch.no_grad():
            logits = nli_model(**inputs).logits
        probs      = torch.softmax(logits, dim=-1)[0]
        nli_score  = probs[2].item()  # Entailment probability

        # Combined score
        combined = lambda_mix * cos_score + (1 - lambda_mix) * nli_score
        reranked.append((chunk_text, combined))

    # Sort descending by combined score
    reranked.sort(key=lambda x: x[1], reverse=True)
    return reranked


def lambda_sensitivity_analysis(
    lambdas: List[float],
    rfg_scores: Dict[float, float]
) -> None:
    """Plot RFG vs lambda to find optimal mixing parameter."""
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(lambdas, [rfg_scores[l] for l in lambdas],
            marker="o", color="#2ECC71", linewidth=2)
    ax.axvline(x=LAMBDA_RERANK, color="red", linestyle="--", alpha=0.7,
               label=f"Default λ={LAMBDA_RERANK}")
    ax.set_xlabel("λ (cosine weight)", fontsize=12)
    ax.set_ylabel("RFG (lower = better)", fontsize=12)
    ax.set_title("RFG vs Re-ranking λ (Sensitivity Analysis)", fontsize=13, fontweight="bold")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("lambda_sensitivity.pdf", dpi=150, bbox_inches="tight")
    plt.show()


print("Re-ranking utilities ready.")

---
## 🏠 PHASE G.3 — Entailment-Similarity Alignment Analysis (§4.5.1, Eq. 3)

In [ ]:
def compute_entailment_similarity_correlation(
    samples: List[QASample],
    model_wrapper: EmbeddingModelWrapper,
    nli_model,
    nli_tokenizer,
    n_samples: int = 200
) -> Dict[str, float]:
    """
    Measure correlation between cosine similarity (retrieval signal)
    and NLI entailment probability (faithfulness signal) (§4.5.1).

    A high correlation = embedding model's similarity scores
    predict logical entailment well → lower expected RFG.
    """
    cos_scores, nli_scores = [], []
    subset = samples[:n_samples]

    for sample in tqdm(subset, desc="Entailment-Similarity Correlation"):
        gold_chunks = chunk_text(sample.gold_context)
        if not gold_chunks: continue
        gold_chunk = gold_chunks[0]

        # Cosine similarity: query ↔ gold chunk
        q_emb    = model_wrapper.encode([sample.question], is_query=True)
        chunk_emb = model_wrapper.encode([gold_chunk], is_query=False)
        cos_sim   = float(np.dot(q_emb[0], chunk_emb[0]))  # Already normalized

        # NLI: P(gold_chunk entails answer)
        inputs = nli_tokenizer(
            gold_chunk, sample.answer,
            return_tensors="pt", truncation=True, max_length=512
        ).to(DEVICE)
        with torch.no_grad():
            logits = nli_model(**inputs).logits
        nli_prob = torch.softmax(logits, dim=-1)[0][2].item()

        cos_scores.append(cos_sim)
        nli_scores.append(nli_prob)

    pearson_r, p_val   = stats.pearsonr(cos_scores, nli_scores)
    spearman_r, sp_val = stats.spearmanr(cos_scores, nli_scores)

    return {
        "pearson_r":  round(pearson_r,  4),
        "spearman_r": round(spearman_r, 4),
        "p_value":    round(p_val,      6),
        "n":          len(cos_scores),
        "cos_scores": cos_scores,
        "nli_scores": nli_scores
    }


print("Mechanistic analysis (entailment-similarity) utilities ready.")

---
## Cell 13 — Statistical Significance Testing (§5.3)
Paired bootstrap resampling, n=10,000 iterations [21].

In [ ]:
def bootstrap_significance(
    scores_a: List[float],
    scores_b: List[float],
    n_bootstrap: int = 10_000,
    alpha: float = 0.05
) -> Dict:
    """
    Paired bootstrap resampling test (§5.3).
    Tests H0: mean(A) == mean(B).
    Following Dror et al. [21].
    """
    assert len(scores_a) == len(scores_b), "Samples must be paired"
    n = len(scores_a)
    observed_diff = np.mean(scores_a) - np.mean(scores_b)
    diffs = []

    rng = np.random.default_rng(SEED)
    for _ in range(n_bootstrap):
        idx    = rng.integers(0, n, size=n)
        boot_a = np.mean(np.array(scores_a)[idx])
        boot_b = np.mean(np.array(scores_b)[idx])
        diffs.append(boot_a - boot_b)

    diffs = np.array(diffs)
    # Two-tailed p-value
    p_value = np.mean(np.abs(diffs) >= np.abs(observed_diff))
    ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])

    return {
        "observed_diff":   round(observed_diff, 4),
        "p_value":         round(p_value, 4),
        "significant":     p_value < alpha,
        "ci_95":           (round(ci_low, 4), round(ci_high, 4)),
        "n_bootstrap":     n_bootstrap
    }


# Example usage
example_a = np.random.normal(0.65, 0.1, 1000).tolist()
example_b = np.random.normal(0.50, 0.1, 1000).tolist()
result = bootstrap_significance(example_a, example_b)
print("Bootstrap test example:")
print(f"  Observed diff:  {result['observed_diff']}")
print(f"  p-value:        {result['p_value']}")
print(f"  Significant:    {result['significant']}")
print(f"  95% CI:         {result['ci_95']}")

---
## 🏠 PHASE H — Visualization & Export (Local CPU, 5 min)

Generates all paper figures from real experimental data.

In [ ]:
PARADIGM_COLORS = {
    "contrastive":       "#E74C3C",
    "multilingual":      "#8E44AD",
    "instruction-tuned": "#27AE60",
}
DATASET_MARKERS = {"NQ": "o", "HotpotQA": "s", "QASPER": "^"}  # circle, square, triangle


def plot_rfg_scatter(df: pd.DataFrame, save_path: str = "rfg_scatter.pdf"):
    """
    Figure 1: Retrieval Quality vs Faithfulness scatter plot.
    Each point = one model × dataset combination.
    Diagonal line = RFG=0 (perfect alignment).
    Points BELOW diagonal = high RFG (poor faithfulness relative to retrieval).
    """
    fig, ax = plt.subplots(figsize=(9, 7))

    # Diagonal: RFG = 0
    lims = [0.35, 0.85]
    ax.plot(lims, lims, "k--", alpha=0.4, linewidth=1.2, label="RFG = 0 (ideal)")

    # Scatter points
    for _, row in df.iterrows():
        color  = PARADIGM_COLORS.get(row["paradigm"], "#555555")
        marker = DATASET_MARKERS.get(row["dataset"], "o")
        ax.scatter(
            row["NDCG@5"], row["mean_faithfulness"],
            color=color, marker=marker, s=120, zorder=3,
            edgecolors="white", linewidth=0.8, alpha=0.9
        )
        ax.annotate(
            row["model"].replace("all-mpnet-base-v2", "SBERT"),
            (row["NDCG@5"], row["mean_faithfulness"]),
            textcoords="offset points", xytext=(6, 3),
            fontsize=7.5, alpha=0.85
        )

    # Legend — paradigm colors
    paradigm_patches = [
        mpatches.Patch(color=c, label=p.title())
        for p, c in PARADIGM_COLORS.items()
    ]
    # Legend — dataset markers
    import matplotlib.lines as mlines
    dataset_handles = [
        mlines.Line2D([0],[0], marker=m, color="gray", linestyle="None",
                      markersize=8, label=d)
        for d, m in DATASET_MARKERS.items()
    ]
    first_legend = ax.legend(
        handles=paradigm_patches + dataset_handles,
        title="Paradigm / Dataset", fontsize=9,
        loc="upper left", framealpha=0.9
    )
    ax.add_artist(first_legend)

    ax.set_xlabel("Retrieval Quality (NDCG@5)", fontsize=12)
    ax.set_ylabel("Faithfulness (Mean AlignScore + NLI)", fontsize=12)
    ax.set_title(
        "Retrieval Quality vs Faithfulness across Embedding Models\n"
        "(Points below diagonal = High RFG)",
        fontsize=13, fontweight="bold"
    )
    ax.set_xlim(lims); ax.set_ylim(0.30, 0.80)
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")


plot_rfg_scatter(SIMULATED_RESULTS)

---
## Figure 2 — RFG Heatmap

In [ ]:
def plot_rfg_heatmap(df: pd.DataFrame, save_path: str = "rfg_heatmap.pdf"):
    """
    Figure 2: RFG heatmap across model × dataset combinations.
    Red = high RFG (bad), Green = low RFG (good).
    Tests H2: HotpotQA should have highest RFG (multi-hop gap).
    """
    pivot = df.pivot(index="model", columns="dataset", values="RFG")
    # Sort models by mean RFG (worst → best)
    pivot["mean_RFG"] = pivot.mean(axis=1)
    pivot = pivot.sort_values("mean_RFG", ascending=False).drop(columns="mean_RFG")

    fig, ax = plt.subplots(figsize=(8, 5))
    sns.heatmap(
        pivot, annot=True, fmt=".3f", cmap="RdYlGn_r",
        ax=ax, linewidths=0.5, linecolor="white",
        vmin=0.0, vmax=0.25,
        cbar_kws={"label": "RFG (↓ better)"}
    )
    ax.set_title(
        "Retrieval-Faithfulness Gap (RFG) by Model × Dataset\n"
        "Red = High Gap (poor faithfulness), Green = Low Gap",
        fontsize=12, fontweight="bold"
    )
    ax.set_xlabel(""); ax.set_ylabel("")
    plt.xticks(rotation=0, fontsize=11)
    plt.yticks(rotation=0, fontsize=9)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")


plot_rfg_heatmap(SIMULATED_RESULTS)

---
## Figure 3 — Paradigm Comparison (H1 test)

In [ ]:
def plot_paradigm_rfg(df: pd.DataFrame, save_path: str = "paradigm_rfg.pdf"):
    """
    Figure 3: Box plot of RFG by training paradigm.
    Tests H1: instruction-tuned < contrastive in RFG.
    """
    fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=True)
    datasets  = ["NQ", "HotpotQA", "QASPER"]

    for ax, dataset in zip(axes, datasets):
        subset = df[df["dataset"] == dataset]
        paradigm_order = ["contrastive", "multilingual", "instruction-tuned"]
        colors = [PARADIGM_COLORS[p] for p in paradigm_order]

        rfg_by_paradigm = [
            subset[subset["paradigm"] == p]["RFG"].values
            for p in paradigm_order
        ]
        bp = ax.boxplot(
            rfg_by_paradigm, labels=[p.replace("-", "\n") for p in paradigm_order],
            patch_artist=True, medianprops={"color": "black", "linewidth": 2}
        )
        for patch, color in zip(bp["boxes"], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.75)

        ax.set_title(dataset, fontsize=13, fontweight="bold")
        ax.set_ylabel("RFG (↓ better)") if ax == axes[0] else None
        ax.grid(True, axis="y", alpha=0.3)
        ax.set_ylim(-0.02, 0.30)

    fig.suptitle(
        "RFG by Embedding Training Paradigm (H1: instruction-tuned < contrastive)",
        fontsize=13, fontweight="bold", y=1.02
    )
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")


plot_paradigm_rfg(SIMULATED_RESULTS)

---
## Results Summary Table (Table 3)

In [ ]:
def generate_results_table(df: pd.DataFrame) -> pd.DataFrame:
    """Generate main results table (Table 3 in paper)."""
    table = df.groupby(["model", "paradigm"]).agg(
        NDCG_5     = ("NDCG@5",           "mean"),
        Recall_5   = ("Recall@5",          "mean"),
        AlignScore = ("align_score",       "mean"),
        NLI_Ent    = ("nli_entailment",    "mean"),
        Faith_Mean = ("mean_faithfulness", "mean"),
        RFG        = ("RFG",               "mean"),
    ).round(3).reset_index()

    # Sort by RFG ascending (best → worst)
    table = table.sort_values("RFG")

    # Styling
    styled = table.style \\n        .background_gradient(subset=["RFG"], cmap="RdYlGn_r", vmin=0, vmax=0.25) \
        .background_gradient(subset=["Faith_Mean"], cmap="RdYlGn", vmin=0.3, vmax=0.75) \
        .set_caption("Table 3: Main Results — Retrieval Quality, Faithfulness, and RFG by Embedding Model") \
        .format(precision=3)
    return table, styled


results_table, results_styled = generate_results_table(SIMULATED_RESULTS)
print("Main Results Table:")
display(results_styled)

---
## Hypothesis Testing Summary (H1-H5)

In [ ]:
def test_all_hypotheses(df: pd.DataFrame) -> pd.DataFrame:
    """
    Test hypotheses H1–H5 from §6 using bootstrap significance testing [21].
    """
    hyp_results = []

    # H1: Instruction-tuned < Contrastive in RFG
    inst_rfg  = df[df["paradigm"]=="instruction-tuned"]["RFG"].tolist()
    cont_rfg  = df[df["paradigm"]=="contrastive"]["RFG"].tolist()
    min_n     = min(len(inst_rfg), len(cont_rfg))
    h1        = bootstrap_significance(cont_rfg[:min_n], inst_rfg[:min_n])
    hyp_results.append({
        "Hypothesis": "H1: Instruction-tuned < Contrastive (RFG)",
        "Observed diff": h1["observed_diff"],
        "p-value": h1["p_value"],
        "Significant": "✓" if h1["significant"] else "✗",
        "Direction": "✓ Supported" if h1["observed_diff"] > 0 and h1["significant"] else "✗ Not supported"
    })

    # H2: HotpotQA has highest RFG
    rfg_by_ds = df.groupby("dataset")["RFG"].mean()
    hyp_results.append({
        "Hypothesis": "H2: HotpotQA has highest RFG (multi-hop)",
        "Observed diff": round(rfg_by_ds.get("HotpotQA", 0) - rfg_by_ds.drop("HotpotQA", errors="ignore").mean(), 4),
        "p-value": "N/A",
        "Significant": "N/A",
        "Direction": "✓ Supported" if rfg_by_ds.idxmax() == "HotpotQA" else "✗ Not supported"
    })

    # H3: Generator independence (compare Llama vs GPT RFG rankings)
    hyp_results.append({
        "Hypothesis": "H3: RFG ranking consistent across generators",
        "Observed diff": "See Spearman correlation",
        "p-value": "TBD",
        "Significant": "TBD",
        "Direction": "Run with real generator data"
    })

    # H4: Entailment-similarity correlation higher for instruction-tuned
    hyp_results.append({
        "Hypothesis": "H4: Instruction-tuned has higher entailment-similarity r",
        "Observed diff": "See §4.5.1 analysis",
        "p-value": "TBD",
        "Significant": "TBD",
        "Direction": "Run mechanistic analysis"
    })

    # H5: Re-ranking reduces RFG by ≥15%
    hyp_results.append({
        "Hypothesis": "H5: Re-ranking reduces worst model RFG by ≥15%",
        "Observed diff": "TBD after re-ranking experiments",
        "p-value": "TBD",
        "Significant": "TBD",
        "Direction": "Run re-ranking pipeline"
    })

    return pd.DataFrame(hyp_results)


hyp_df = test_all_hypotheses(SIMULATED_RESULTS)
print("Hypothesis Testing Summary:")
display(hyp_df)

---
## Export All Figures & Results

In [ ]:
import os

OUTPUT_DIR = 'paper_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Generate figures from real results (or simulated if Phase G not done)
df = results_df if results_df is not None else SIMULATED_RESULTS

plot_rfg_scatter(df,  save_path=f'{OUTPUT_DIR}/fig1_rfg_scatter.pdf')
plot_rfg_heatmap(df,  save_path=f'{OUTPUT_DIR}/fig2_rfg_heatmap.pdf')
plot_paradigm_rfg(df, save_path=f'{OUTPUT_DIR}/fig3_paradigm_rfg.pdf')

df.to_csv(f'{OUTPUT_DIR}/full_results.csv', index=False)

# Cost report
print(f'\n{cost_tracker.summary()}')

print(f'\nAll outputs saved to ./{OUTPUT_DIR}/')
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(f'{OUTPUT_DIR}/{f}')
    print(f'  {f:<40s} {size:>8,} bytes')

---
## Execution Checklist

| # | Phase | Where | Status |
|---|-------|-------|--------|
| 1 | **A** Embedding + FAISS | 🏠 Local overnight | ☐ |
| 2 | **B** Retrieval quality | 🏠 Local (30min) | ☐ |
| 3 | **C** GPT-4o-mini gen | 🏠 Local + API ($5) | ☐ |
| 4 | **D** DeBERTa-NLI | 🏠 Local overnight | ☐ |
| 5 | **E** AlignScore | ☁️ Colab T4 (1-2h) | ☐ |
| 6 | **F** Llama-3 subset | ☁️ Colab T4 (3-4h) | ☐ |
| 7 | **G** RFG + Analysis | 🏠 Local (2-3h) | ☐ |
| 8 | **H** Figures + Export | 🏠 Local (5min) | ☐ |

### Moving checkpoints between Local ↔ Colab

```bash
# Upload local checkpoints to Google Drive for Colab phases
# Option 1: Google Drive desktop app (drag rag_faith_checkpoints/ folder)
# Option 2: rclone
rclone copy ./rag_faith_checkpoints/ gdrive:rag_faith_checkpoints/
```

On Colab, the notebook auto-mounts Drive and reads checkpoints from  
`/content/drive/MyDrive/rag_faith_checkpoints/`

### Quick test mode
Set `N_QUERIES = 50` in Cell 2 to do a fast end-to-end sanity check  
before committing to the full 1,000-query run.

### References implemented:
[1] Lewis et al. 2020 (RAG) | [2] E5-instruct | [3] BGE-M3 | [4] SBERT  
[6] Sun et al. ReDeEP (attention analysis) | [7] Chen et al. Each to Their Own  
[9] Tamber et al. FaithJudge | [14] FAISS | [16] AlignScore  
[17] NQ | [18] HotpotQA | [19] QASPER | [20] DeBERTa-NLI  
[21] Bootstrap significance testing
